In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer

train_path = os.path.abspath("../input/siim-isic-melanoma-classification/train.csv")
train_df = pd.read_csv(train_path)

target_col = "target"
global_mean = train_df[target_col].mean()

numeric_features = ["age_approx"]
# expand one‑hot to include low‑cardinality categorical columns
categorical_features = ["sex", "diagnosis", "anatom_site_general_challenge"]

site_means = train_df.groupby("anatom_site_general_challenge")[target_col].mean()
train_df["site_mean"] = train_df["anatom_site_general_challenge"].map(site_means)

diagnosis_means = train_df.groupby("diagnosis")[target_col].mean()
train_df["diagnosis_mean"] = train_df["diagnosis"].map(diagnosis_means)

patient_means = train_df.groupby("patient_id")[target_col].mean()
train_df["patient_mean"] = train_df["patient_id"].map(patient_means)

site_diag_means = (
    train_df.groupby(["anatom_site_general_challenge", "diagnosis"])[target_col]
    .mean()
    .reset_index(name="site_diag_mean")
)
train_df = train_df.merge(
    site_diag_means,
    on=["anatom_site_general_challenge", "diagnosis"],
    how="left",
)

for col in ["site_mean", "diagnosis_mean", "patient_mean", "site_diag_mean"]:
    train_df[col].fillna(global_mean, inplace=True)

numeric_features.extend(
    ["site_mean", "diagnosis_mean", "patient_mean", "site_diag_mean"]
)

numeric_transformer = SimpleImputer(strategy="median")
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "clf",
            LogisticRegressionCV(
                Cs=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0],
                cv=5,
                scoring="roc_auc",
                max_iter=1000,
                n_jobs=5,
                solver="lbfgs",
                class_weight="balanced",
                refit=True,
            ),
        ),
    ]
)

X = train_df[numeric_features + categorical_features]
y = train_df[target_col]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

val_pred = model.predict_proba(X_val)[:, 1]

baseline_val = (
    X_val["site_mean"]
    + X_val["diagnosis_mean"]
    + X_val["patient_mean"]
    + X_val["site_diag_mean"]
) / 4.0

val_auc = roc_auc_score(y_val, val_pred)

best_auc = val_auc
best_weight = 1.0  # weight for model prediction

for w in np.arange(0.0, 1.001, 0.001):
    blended = w * val_pred + (1 - w) * baseline_val
    auc = roc_auc_score(y_val, blended)
    if auc > best_auc:
        best_auc = auc
        best_weight = w

print(f"Validation AUC (model only): {val_auc:.5f}")
print(f"Best blended AUC: {best_auc:.5f} with weight {best_weight:.3f}")

model.fit(X, y)

blend_weight = best_weight



/tmp/ipykernel_11/3143979619.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df[col].fillna(global_mean, inplace=True)


Validation AUC (model only): 1.00000
Best blended AUC: 1.00000 with weight 0.063


In [2]:
test_meta_path = os.path.abspath("../input/siim-isic-melanoma-classification/test.csv")
test_meta = pd.read_csv(test_meta_path)

for col in ["diagnosis", "benign_malignant"]:
    if col not in test_meta.columns:
        test_meta[col] = np.nan

test_meta["diagnosis"] = test_meta["diagnosis"].astype(str)

test_meta["site_mean"] = test_meta["anatom_site_general_challenge"].map(site_means)
test_meta["site_mean"].fillna(global_mean, inplace=True)

test_meta["diagnosis_mean"] = test_meta["diagnosis"].map(diagnosis_means)
test_meta["diagnosis_mean"].fillna(global_mean, inplace=True)

test_meta["patient_mean"] = test_meta["patient_id"].map(patient_means)
test_meta["patient_mean"].fillna(global_mean, inplace=True)

test_meta = test_meta.merge(
    site_diag_means,
    on=["anatom_site_general_challenge", "diagnosis"],
    how="left",
)
test_meta["site_diag_mean"].fillna(global_mean, inplace=True)

X_test = test_meta[numeric_features + categorical_features]

test_meta["model_pred"] = model.predict_proba(X_test)[:, 1]

baseline_test = (
    test_meta["site_mean"]
    + test_meta["diagnosis_mean"]
    + test_meta["patient_mean"]
    + test_meta["site_diag_mean"]
) / 4.0

test_meta["pred_target"] = (
    blend_weight * test_meta["model_pred"] + (1 - blend_weight) * baseline_test
)



/tmp/ipykernel_11/3148158111.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_meta["site_mean"].fillna(global_mean, inplace=True)
/tmp/ipykernel_11/3148158111.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)'

In [3]:
sample_sub_path = os.path.abspath(
    "../input/siim-isic-melanoma-classification/sample_submission.csv"
)
sub = pd.read_csv(sample_sub_path)

sub = sub.merge(test_meta[["image_name", "pred_target"]], on="image_name", how="left")
sub["target"] = sub["pred_target"].fillna(global_mean)
sub.drop(columns=["pred_target"], inplace=True)



In [4]:
output_path = "submission.csv"
sub.to_csv(output_path, index=False)
print(f"Submission written to {output_path}, shape: {sub.shape}")

Submission written to submission.csv, shape: (4142, 2)
